**Reading note.** This notebook records the Week 6 analysis. Later held-out tests did not find a consistent forecasting advantage for the MPS gauges. See [the final findings](../week7/FINDINGS.md) before interpreting these charts as risk signals.


# What comes out of the MPS? A guided tour of the model's outputs

**Quantum-inspired market regime & tail-complexity module — companion to the Week-6 Orthogonality Proof**

The orthogonality notebook shows the *gauges* are non-redundant. This notebook answers the more
basic question people ask first: **when we fit the MPS to a window of returns, what do we actually
get out of it?**

Short answer: a full **joint probability distribution** over the day's return vector — something a
covariance matrix is not. Everything below is a view of that one object:

1. the fitted tensor itself (and how compact it is),
2. model-implied **marginals** per asset,
3. model-implied **volatility and correlation** (the covariance world, recovered as a special case),
4. **pairwise joint tables** and crash **conditionals**,
5. **tail dependence** and the **joint-crash corner** (beyond-covariance content),
6. **tail asymmetry** — the Fear-vs-Euphoria reading,
7. **simulated scenarios** (the model as a generator),
8. how one fit becomes one **row of the weekly gauge panel**.

Model family and settings match the rolling battery behind the gauge panel: quantile bins d=8,
bond dimension chi=8, DMRG training. We fit two 250-day windows of the GFC book for contrast:
a **calm** window (ending 2019-06-28) and the **COVID crash** window (ending 2020-04-01).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, pandas as pd
from mps_regime.data import to_returns
from mps_regime import binning, dmrg

pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

prices = pd.read_parquet('../data_external/combined_gfc_2026.parquet').astype(float)
returns = to_returns(prices, kind='log')
print('GFC book:', returns.shape[1], 'assets,', len(returns), 'days,',
      returns.index.min().date(), '->', returns.index.max().date())
returns.tail(3)

GFC book: 12 assets, 4834 days, 2007-04-13 -> 2026-06-30


,USMKT,AAPL,NVDA,IWM,EFA,EEM,TLT,LQD,HYG,VNQ,GLD,DBC
2026-06-26,-0.0005,0.0309,-0.0165,0.0031,-0.0059,-0.0114,0.0001,0.0000,-0.0006,0.0151,0.0112,-0.0135
2026-06-29,0.0117,-0.0072,0.0126,-0.0029,0.0088,0.0036,0.0010,0.0018,0.0023,-0.0053,-0.0136,-0.0004
2026-06-30,0.0079,0.0267,0.0259,0.0049,0.0041,0.0144,-0.0118,-0.0058,-0.0005,-0.0177,-0.0005,0.0038


## 1 — The input: one rolling window of daily log returns

The model sees exactly what a covariance estimator would see: a 250-day window of the 12-asset
return panel. No options data, no volume, no external features.

In [2]:
END_CALM, END_CRISIS = '2019-06-28', '2020-04-01'
win_calm  = returns.loc[:END_CALM].tail(250)
win_crisis = returns.loc[:END_CRISIS].tail(250)
print('calm window  :', win_calm.index.min().date(), '->', win_calm.index.max().date())
print('crisis window:', win_crisis.index.min().date(), '->', win_crisis.index.max().date())
win_crisis.tail(3)

calm window  : 2018-07-02 -> 2019-06-28
crisis window: 2019-04-05 -> 2020-04-01


,USMKT,AAPL,NVDA,IWM,EFA,EEM,TLT,LQD,HYG,VNQ,GLD,DBC
2020-03-30,0.0286,0.0281,0.0496,0.0218,0.0200,0.0164,-0.0083,0.0121,0.0085,0.0216,0.0044,-0.0026
2020-03-31,-0.0124,-0.0020,-0.0075,-0.0051,-0.0088,0.0068,-0.0081,-0.0033,-0.0080,-0.0245,-0.0324,-0.0124
2020-04-01,-0.0461,-0.0541,-0.0811,-0.0706,-0.0453,-0.0437,0.0139,-0.0146,-0.0278,-0.0680,0.0094,-0.0289


## 2 — Discretization: returns become bin symbols

Each asset's returns are cut into **d = 8 quantile bins** on the training window (bin 0 = worst
days, bin 7 = best days). The MPS learns the joint distribution of these 12 bin symbols. The bin
edges below ARE part of the model — they are refit each window, so "bin 0" always means "that
asset's own worst 12.5% of days in this window".

In [3]:
D_BINS, CHI, SWEEPS = 8, 8, 16
edges = binning.bin_edges(win_crisis, n_bins=D_BINS)
edges_df = pd.DataFrame({a: e for a, e in edges.items()},
                        index=[f'edge {i}' for i in range(1, D_BINS)]).T
print('interior bin edges per asset (daily log-return cut points), crisis window:')
edges_df

interior bin edges per asset (daily log-return cut points), crisis window:


,edge 1,edge 2,edge 3,edge 4,edge 5,edge 6,edge 7
USMKT,-0.0093,-0.0038,-0.0011,0.0007,0.0028,0.0058,0.0089
AAPL,-0.0185,-0.0073,-0.0025,0.0007,0.0066,0.0123,0.0196
NVDA,-0.0268,-0.0152,-0.0054,0.0019,0.0085,0.0154,0.0298
IWM,-0.0130,-0.0062,-0.0029,0.0003,0.0031,0.0069,0.0121
EFA,-0.0091,-0.0043,-0.0014,0.0008,0.0029,0.0051,0.0086
EEM,-0.0130,-0.0067,-0.0020,0.0009,0.0042,0.0072,0.0112
TLT,-0.0090,-0.0049,-0.0013,0.0015,0.0034,0.0070,0.0110
LQD,-0.0034,-0.0014,-0.0003,0.0008,0.0017,0.0026,0.0045
HYG,-0.0035,-0.0014,-0.0006,0.0003,0.0010,0.0021,0.0034
VNQ,-0.0130,-0.0048,-0.0010,0.0015,0.0035,0.0064,0.0096


In [4]:
binned = binning.apply_bins(win_crisis, edges)
print('the last 5 days of the crisis window, as bin symbols (0 = worst octile ... 7 = best):')
binned.tail()

the last 5 days of the crisis window, as bin symbols (0 = worst octile ... 7 = best):


,USMKT,AAPL,NVDA,IWM,EFA,EEM,TLT,LQD,HYG,VNQ,GLD,DBC
2020-03-26,7,7,7,7,7,7,5,4,7,7,7,2
2020-03-27,0,0,1,0,0,0,7,1,6,3,1,2
2020-03-30,7,7,7,7,7,7,1,7,7,7,5,3
2020-03-31,0,3,2,2,1,5,1,1,0,0,0,0
2020-04-01,0,0,0,0,0,0,7,0,0,0,7,0


## 3 — The fitted object: a matrix-product state over 12 assets

`fit_dmrg` returns an `MPSFit`: one small tensor per asset, chained by bonds of dimension chi=8.
The chain of tensors *is* the joint distribution: P(x1,...,x12) = |psi(x1,...,x12)|^2 / Z, where
psi is the product of the tensors. Note the compression — the full joint table over 8 bins x 12
assets would have ~6.9e10 entries; the MPS stores a few thousand parameters.

*(This cell is the only training in the notebook — two DMRG fits, a few seconds each on the pod.)*

In [5]:
fit_crisis = dmrg.fit_dmrg(win_crisis, d=D_BINS, chi=CHI, sweeps=SWEEPS, seed=0)
fit_calm   = dmrg.fit_dmrg(win_calm,   d=D_BINS, chi=CHI, sweeps=SWEEPS, seed=0)

shapes = [tuple(c.shape) for c in fit_crisis.cores]
n_params = sum(int(np.prod(s)) for s in shapes)
full_table = D_BINS ** len(shapes)
summary = pd.DataFrame({
    'value': [len(shapes), D_BINS, CHI, str(shapes[0]) + ' ... ' + str(shapes[-1]),
              f'{n_params:,}', f'{full_table:.2e}', f'{full_table / n_params:.1e}x']},
    index=['assets (tensors in the chain)', 'bins per asset (d)', 'bond dimension (chi)',
           'first / last core shape', 'total parameters', 'full joint-table size',
           'compression ratio'])
summary

,value
assets (tensors in the chain),12
bins per asset (d),8
bond dimension (chi),8
first / last core shape,"(8, 8) ... (8, 8)"
total parameters,"5,248"
full joint-table size,6.87e+10
compression ratio,1.3e+07x


## 4 — Output 1: model-implied marginals per asset

`one_site_marginals()` contracts the chain into P(asset i in bin b) — an exact model quantity, not
a sample. Rows sum to 1. Comparing with the empirical bin frequencies (which are ~1/8 by
construction of quantile bins) shows how much probability the *fitted* model moves toward the
tails: during the crisis fit, extra mass sits in the extreme bins.

In [6]:
marg = pd.DataFrame(fit_crisis.one_site_marginals(), index=win_crisis.columns,
                    columns=[f'bin {b}' for b in range(D_BINS)])
marg['sum'] = marg.sum(axis=1)
print('P(asset lands in bin b) implied by the crisis-window fit:')
marg

P(asset lands in bin b) implied by the crisis-window fit:


,bin 0,bin 1,bin 2,bin 3,bin 4,bin 5,bin 6,bin 7,sum
USMKT,0.1280,0.1240,0.1240,0.1240,0.1240,0.1240,0.1240,0.1280,1.0000
AAPL,0.1280,0.1240,0.1240,0.1240,0.1240,0.1240,0.1240,0.1280,1.0000
NVDA,0.1224,0.1254,0.1252,0.1236,0.1234,0.1312,0.1242,0.1247,1.0000
IWM,0.1252,0.1289,0.1268,0.1238,0.1276,0.1223,0.1233,0.1222,1.0000
EFA,0.1204,0.1268,0.1187,0.1277,0.1336,0.1283,0.1241,0.1204,1.0000
EEM,0.1199,0.1240,0.1284,0.1280,0.1231,0.1250,0.1288,0.1228,1.0000
TLT,0.1260,0.1197,0.1301,0.1271,0.1213,0.1296,0.1210,0.1252,1.0000
LQD,0.1183,0.1252,0.1264,0.1160,0.1218,0.1240,0.1254,0.1429,1.0000
HYG,0.1181,0.1277,0.1280,0.1308,0.1326,0.1208,0.1225,0.1194,1.0000
VNQ,0.1253,0.1140,0.1338,0.1158,0.1313,0.1202,0.1263,0.1333,1.0000


## 5 — Output 2: the covariance world, recovered as a special case

The MPS contains the second-moment information a risk desk already uses — `mps_volatility()` and
`mps_correlation()` read annualized vols and the correlation matrix straight off the fitted
distribution. This is the sanity check that the model *contains* the classical view before going
beyond it.

In [7]:
vols = pd.DataFrame({
    'MPS-implied vol (ann.)': fit_crisis.mps_volatility(annualize=True),
    'empirical vol (ann.)': win_crisis.std().to_numpy() * np.sqrt(252),
}, index=win_crisis.columns)
vols

,MPS-implied vol (ann.),empirical vol (ann.)
USMKT,0.2296,0.3000
AAPL,0.3379,0.3944
NVDA,0.4663,0.5353
IWM,0.2594,0.3349
EFA,0.1987,0.2741
EEM,0.2401,0.3107
TLT,0.1756,0.2128
LQD,0.1042,0.1574
HYG,0.0966,0.1414
VNQ,0.2570,0.3607


In [8]:
corr_mps = pd.DataFrame(fit_crisis.mps_correlation(), index=win_crisis.columns,
                        columns=win_crisis.columns)
corr_emp = win_crisis.corr()
print('model-implied vs empirical correlation, first 5 assets:')
pd.concat({'MPS': corr_mps.iloc[:5, :5], 'empirical': corr_emp.iloc[:5, :5]}, axis=1)

model-implied vs empirical correlation, first 5 assets:


MPS                             empirical                            
       USMKT   AAPL   NVDA    IWM    EFA     USMKT   AAPL   NVDA    IWM    EFA
USMKT 1.0000 0.7399 0.7334 0.8089 0.7906    1.0000 0.8829 0.8302 0.9637 0.9619
AAPL  0.7399 1.0000 0.6323 0.6698 0.6537    0.8829 1.0000 0.7911 0.8231 0.8298
NVDA  0.7334 0.6323 1.0000 0.7124 0.6904    0.8302 0.7911 1.0000 0.8164 0.8031
IWM   0.8089 0.6698 0.7124 1.0000 0.8112    0.9637 0.8231 0.8164 1.0000 0.9362
EFA   0.7906 0.6537 0.6904 0.8112 1.0000    0.9619 0.8298 0.8031 0.9362 1.0000

## 6 — Output 3: pairwise joint tables and crash conditionals

Here the model starts saying things a correlation matrix cannot. `two_site_marginal(i, j)` is the
exact 8x8 joint probability table for a pair of assets. The bottom-left corner is the joint-crash
cell. From it: **P(asset j crashes | asset i crashes)** — the number a risk manager actually wants —
versus the unconditional crash probability.

In [9]:
i, j = 0, 1
joint = pd.DataFrame(fit_crisis.two_site_marginal(i, j),
                     index=[f'{win_crisis.columns[i]} bin {b}' for b in range(D_BINS)],
                     columns=[f'{win_crisis.columns[j]} bin {b}' for b in range(D_BINS)])
print(f'exact joint table P({win_crisis.columns[i]} bin, {win_crisis.columns[j]} bin), crisis fit:')
joint

exact joint table P(USMKT bin, AAPL bin), crisis fit:


,AAPL bin 0,AAPL bin 1,AAPL bin 2,AAPL bin 3,AAPL bin 4,AAPL bin 5,AAPL bin 6,AAPL bin 7
USMKT bin 0,0.0960,0.0160,0.0040,0.0040,0.0040,0.0000,-0.0000,0.0040
USMKT bin 1,0.0160,0.0480,0.0320,0.0160,0.0040,0.0000,0.0040,0.0040
USMKT bin 2,0.0040,0.0280,0.0400,0.0200,0.0120,0.0080,0.0040,0.0080
USMKT bin 3,0.0120,0.0120,0.0080,0.0320,0.0360,0.0160,0.0040,0.0040
USMKT bin 4,0.0000,0.0080,0.0160,0.0200,0.0400,0.0200,0.0160,0.0040
USMKT bin 5,0.0000,0.0040,0.0120,0.0160,0.0160,0.0400,0.0280,0.0080
USMKT bin 6,0.0000,0.0040,0.0080,0.0120,0.0120,0.0360,0.0320,0.0200
USMKT bin 7,-0.0000,0.0040,0.0040,0.0040,-0.0000,0.0040,0.0360,0.0760


In [10]:
def crash_conditional(fit, cols, i, j, k_bins=1):
    T = fit.two_site_marginal(i, j)
    p_i = T[:k_bins, :].sum(); p_j = T[:, :k_bins].sum(); p_ij = T[:k_bins, :k_bins].sum()
    return {'pair': f'{cols[i]} & {cols[j]}',
            'P(j crash)': p_j, 'P(j crash | i crash)': p_ij / p_i,
            'dependence multiple': (p_ij / p_i) / p_j}

rows = [crash_conditional(f, win_crisis.columns, a, b)
        for f in (fit_calm, fit_crisis) for a, b in [(0, 1), (0, 5), (3, 7)]]
cc = pd.DataFrame(rows, index=['calm'] * 3 + ['crisis'] * 3)
print('crash = bottom bin (worst 12.5% of own days). Multiple > 1 = crashes attract each other:')
cc

crash = bottom bin (worst 12.5% of own days). Multiple > 1 = crashes attract each other:


,pair,P(j crash),P(j crash | i crash),dependence multiple
calm,USMKT & AAPL,0.1280,0.6875,5.3711
calm,USMKT & EEM,0.1324,0.5825,4.3980
calm,IWM & LQD,0.1331,0.1021,0.7666
crisis,USMKT & AAPL,0.1280,0.7500,5.8594
crisis,USMKT & EEM,0.1199,0.5923,4.9402
crisis,IWM & LQD,0.1183,0.1853,1.5664


## 7 — Output 4: tail dependence and the joint-crash corner (beyond covariance)

`mps_tail_dependence(q)` computes the pairwise lower-tail dependence matrix from the fitted
distribution, and the **k-way corner** — the probability that 3+ of the 12 assets sit in their
bottom bin on the same day — is the Joint-Crash gauge (F14). We estimate it from 40,000 exact
model samples and compare with the independence baseline: this is dependence the covariance matrix
understates and independence misses entirely.

In [11]:
taildep = pd.DataFrame(fit_crisis.mps_tail_dependence(q=1 / D_BINS),
                       index=win_crisis.columns, columns=win_crisis.columns)
print('pairwise lower-tail dependence (crisis fit), first 6 assets:')
taildep.iloc[:6, :6]

pairwise lower-tail dependence (crisis fit), first 6 assets:


,USMKT,AAPL,NVDA,IWM,EFA,EEM
USMKT,1.0000,0.7680,0.9002,0.8187,0.9124,0.8266
AAPL,0.7680,1.0000,0.8188,0.6948,0.8238,0.7446
NVDA,0.9002,0.8188,1.0000,0.9181,1.4417,1.2474
IWM,0.8187,0.6948,0.9181,1.0000,0.8994,0.8215
EFA,0.9124,0.8238,1.4417,0.8994,1.0000,1.5112
EEM,0.8266,0.7446,1.2474,0.8215,1.5112,1.0000


In [12]:
def kway_corner(fit, k=3, n=40_000):
    bins_ = fit.sample_bins(n, seed=7)
    return float(((bins_ == 0).sum(axis=1) >= k).mean())

marg_calm, marg_crisis = fit_calm.one_site_marginals(), fit_crisis.one_site_marginals()

def indep_kway(marg, k=3, n=40_000, seed=11):
    rng = np.random.default_rng(seed)
    p0 = marg[:, 0]
    hits = rng.random((n, len(p0))) < p0[None, :]
    return float((hits.sum(axis=1) >= k).mean())

corner = pd.DataFrame({
    'P(>=3 of 12 in bottom bin)': [kway_corner(fit_calm), kway_corner(fit_crisis)],
    'independence baseline': [indep_kway(marg_calm), indep_kway(marg_crisis)],
}, index=['calm fit', 'crisis fit'])
corner['dependence lift'] = corner.iloc[:, 0] / corner.iloc[:, 1]
corner

,P(>=3 of 12 in bottom bin),independence baseline,dependence lift
calm fit,0.2192,0.2052,1.0682
crisis fit,0.1837,0.1796,1.0230


## 8 — Output 5: tail asymmetry — the Fear-vs-Euphoria reading (F12)

Same construction on both corners: joint-**crash** mass (bottom bins) minus joint-**rally** mass
(top bins). A symmetric model — any Gaussian — scores exactly zero here by construction. This is
the headline gauge of the orthogonality proof, read straight off the fit.

In [13]:
def corner_masses(fit, k=3, n=40_000):
    b = fit.sample_bins(n, seed=7)
    lower = float(((b == 0).sum(axis=1) >= k).mean())
    upper = float(((b == D_BINS - 1).sum(axis=1) >= k).mean())
    return lower, upper, lower - upper

asym = pd.DataFrame([corner_masses(fit_calm), corner_masses(fit_crisis)],
                    columns=['joint-crash mass', 'joint-rally mass', 'asymmetry (F12 direction)'],
                    index=['calm fit', 'crisis fit'])
print('positive asymmetry = co-crash dominance (fear); negative = co-rally (euphoria):')
asym

positive asymmetry = co-crash dominance (fear); negative = co-rally (euphoria):


,joint-crash mass,joint-rally mass,asymmetry (F12 direction)
calm fit,0.2192,0.2026,0.0166
crisis fit,0.1837,0.2107,-0.0270


## 9 — Output 6: the model as a scenario generator

`sample_returns(n)` draws whole cross-sectional days from the fitted joint distribution —
bin symbols first, then a return inside each bin. Each row is a coherent simulated market day
carrying the window's dependence structure, including the co-crash days a Gaussian generator
underproduces. (Week-8 caveat: for production scenario use, recalibrate marginals to the
empirical window.)

In [14]:
scen = pd.DataFrame(fit_crisis.sample_returns(5, seed=42), columns=win_crisis.columns)
print('five simulated market days from the crisis fit (daily log returns):')
scen

five simulated market days from the crisis fit (daily log returns):


,USMKT,AAPL,NVDA,IWM,EFA,EEM,TLT,LQD,HYG,VNQ,GLD,DBC
0,0.0073,0.0546,0.0089,-0.0011,0.0045,-0.0005,0.0050,0.0008,-0.0004,0.0068,-0.0031,0.0033
1,0.0002,0.0021,0.0142,0.0009,-0.0003,0.0011,0.0216,-0.0009,0.0022,0.0044,-0.0292,0.0046
2,0.0070,0.0146,0.0209,0.0021,0.0161,0.0036,-0.0038,-0.0006,0.0073,0.0013,0.0057,0.0019
3,0.0041,-0.0044,0.0213,0.0093,0.0058,0.0022,-0.0242,-0.0192,-0.0008,0.0028,0.0045,-0.0094
4,-0.0434,-0.0762,-0.0900,-0.0773,-0.0217,-0.0807,0.0364,0.0030,0.0003,-0.0114,0.0043,-0.0027


In [15]:
big = fit_crisis.sample_returns(20_000, seed=1)
sim_stats = pd.DataFrame({
    'sim mean': big.mean(0), 'sim vol (ann.)': big.std(0) * np.sqrt(252),
    'sim skew': pd.DataFrame(big).skew(), 'window skew': win_crisis.skew().to_numpy(),
}, index=win_crisis.columns)
sim_stats.head(6)

,sim mean,sim vol (ann.),sim skew,window skew
USMKT,-0.0009,0.3031,NaN,-1.4934
AAPL,0.0009,0.3942,NaN,-0.5974
NVDA,0.0012,0.5376,NaN,-0.5476
IWM,-0.0015,0.3318,NaN,-2.2140
EFA,-0.0009,0.2677,NaN,-2.1973
EEM,-0.0009,0.3054,NaN,-2.0143


## 10 — From one fit to the weekly gauge panel

The rolling battery repeats this fit every 10 trading days on a sliding 250-day window and reduces
each fit to the scalar gauges — that is the `gauge_panel.csv` behind the orthogonality proof and
the dashboards. Below: the committed panel rows nearest our two windows. The columns are exactly
the quantities computed above (plus the classical panel used as the redundancy bar).

*A note on what we do NOT read off the fit:* entanglement entropy and bond-dimension diagnostics
were withdrawn as market features after external review — they are functionals of the signed state,
which the likelihood does not identify. Everything shown in this notebook is a probability
functional, which is identified.

In [16]:
panel = pd.read_csv('results/gauge_panel.csv', parse_dates=['date']).set_index('date')
sel = panel.loc[['2019-06-28' in str(d) or '2020-0' in str(d) for d in panel.index]]
nearest = panel.iloc[[panel.index.get_indexer([pd.Timestamp(END_CALM)], method='nearest')[0],
                      panel.index.get_indexer([pd.Timestamp(END_CRISIS)], method='nearest')[0]]]
cols = ['G_asymmetry', 'G_jointcrash', 'G_taildep', 'G_complexity_HO', 'p_stress',
        'realized_vol', 'avg_corr', 'VIX']
print('committed gauge-panel rows nearest the two windows:')
nearest[cols]

committed gauge-panel rows nearest the two windows:


,G_asymmetry,G_jointcrash,G_taildep,G_complexity_HO,p_stress,realized_vol,avg_corr,VIX
date,,,,,,,,
2019-06-25,0.0208,0.0058,1.7502,-4.2731,0.0414,0.1158,0.2964,16.4069
2020-03-26,0.0716,0.0183,1.9908,-3.6376,1.0000,0.2087,0.4412,18.4959


## Summary — the model's output surface

| Output | Method | What it gives a risk desk |
|---|---|---|
| Joint distribution | the fit itself | the object covariance only approximates |
| Marginals | `one_site_marginals()` | per-asset bin probabilities, exact |
| Vol & correlation | `mps_volatility/correlation()` | the classical view, recovered |
| Pair joint tables | `two_site_marginal(i,j)` | crash conditionals: P(j crash \| i crash) |
| Tail dependence | `mps_tail_dependence(q)` | pairwise co-crash strength beyond correlation |
| Joint-crash corner | samples / corner mass | the Joint-Crash gauge (F14) |
| Tail asymmetry | corner difference | the Fear-vs-Euphoria gauge (F12) |
| Scenarios | `sample_returns(n)` | simulated days with realistic co-crash mass |

One fit -> one row of the weekly gauge panel; 434 windows -> the orthogonality proof.